# TabPFN Cross-Grid Transfer: Train Small → Predict IEEE 300

**Goal**: Train on IEEE grids (9, 14, 30, 39, 57, 118) → predict IEEE 300 (unseen).

**Constraint**: No AC power flow on test grid for features (only for evaluation ground truth).

**Training**: 6 grids × 15 loading scales (0.50–1.30) ≈ 4000 bus-rows  
**Test**: case300 at scale=1.0 only (300 buses)

In [1]:
import copy
import pandapower as pp
import pandapower.networks as pn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

## 1. Physics Functions: FDPF + Feature Extraction

In [2]:
def compute_ybus_dc_angles(net):
    """
    Compute DC-like angles using the FULL Y-bus B' matrix.
    Unlike pandapower's DC PF which uses a simplified B matrix,
    this uses the actual admittance matrix including transformer models.
    Returns angles in degrees (pandapower bus ordering).
    """
    from pandapower.pd2ppc import _pd2ppc
    from pandapower.auxiliary import _init_runpp_options
    from pandapower.pypower.makeYbus import makeYbus

    n = len(net.bus)
    bus_list = net.bus.index.tolist()
    bus_to_idx = {b: i for i, b in enumerate(bus_list)}
    S_base = 100.0

    net_copy = copy.deepcopy(net)
    _init_runpp_options(net_copy, algorithm='nr', calculate_voltage_angles=True,
                        init='flat', max_iteration=1, tolerance_mva=1e-8,
                        trafo_model='t', trafo_loading='current',
                        enforce_q_lims=False, check_connectivity=False,
                        voltage_depend_loads=False, consider_line_temperature=False,
                        distributed_slack=False)
    ppc, ppci = _pd2ppc(net_copy)
    Ybus_sparse, _, _ = makeYbus(ppci['baseMVA'], ppci['bus'], ppci['branch'])
    Ybus = Ybus_sparse.toarray()
    bus_lookup = net_copy._pd2ppc_lookups['bus']
    pp_to_ppc = bus_lookup
    n_ppc = Ybus.shape[0]

    # Identify bus types
    ext_set = set(net.ext_grid['bus'].values)
    pv_set = (set(net.gen['bus'].values) if len(net.gen) > 0 else set()) - ext_set
    slack_ppc = [pp_to_ppc[bus_to_idx[b]] for b in ext_set]
    pv_ppc = [pp_to_ppc[bus_to_idx[b]] for b in pv_set]
    pq_ppc = [pp_to_ppc[bus_to_idx[b]] for b in bus_list
              if b not in ext_set and b not in pv_set]
    non_slack_ppc = sorted(set(pv_ppc + pq_ppc))

    # Build P injection vector
    p_spec = np.zeros(n_ppc)
    for row_idx in net.load.index:
        load = net.load.loc[row_idx]
        p_spec[pp_to_ppc[bus_to_idx[load['bus']]]] -= load['p_mw'] / S_base
    if len(net.gen) > 0:
        for row_idx in net.gen.index:
            gen = net.gen.loc[row_idx]
            p_spec[pp_to_ppc[bus_to_idx[gen['bus']]]] += gen['p_mw'] / S_base
    if len(net.shunt) > 0:
        for row_idx in net.shunt.index:
            shunt = net.shunt.loc[row_idx]
            p_spec[pp_to_ppc[bus_to_idx[shunt['bus']]]] -= shunt['p_mw'] / S_base

    # B' = -Im(Y_bus) — includes full transformer models
    Bp = -Ybus.imag
    Bp_sub = Bp[np.ix_(non_slack_ppc, non_slack_ppc)]
    p_sub = p_spec[non_slack_ppc]

    # Solve: B' * theta = P  (DC approximation with full Y-bus)
    try:
        theta_sub = np.linalg.solve(Bp_sub, p_sub)
    except np.linalg.LinAlgError:
        theta_sub = np.linalg.lstsq(Bp_sub, p_sub, rcond=None)[0]

    theta_ppc = np.zeros(n_ppc)
    for i, ppc_idx in enumerate(non_slack_ppc):
        theta_ppc[ppc_idx] = theta_sub[i]

    # Map back to pandapower bus ordering
    theta_pp = np.zeros(n)
    for pp_idx in range(n):
        theta_pp[pp_idx] = np.degrees(theta_ppc[pp_to_ppc[pp_idx]])

    return theta_pp


def compute_fdpf_vq(net, theta_dc_deg, n_iter=10):
    """
    Fast Decoupled Power Flow using pandapower's internal Y-bus.
    Returns (V, Q_bus_mvar) in pandapower convention (positive = consumed).
    """
    from pandapower.pd2ppc import _pd2ppc
    from pandapower.auxiliary import _init_runpp_options
    from pandapower.pypower.makeYbus import makeYbus

    n = len(net.bus)
    bus_list = net.bus.index.tolist()
    bus_to_idx = {b: i for i, b in enumerate(bus_list)}
    S_base = 100.0

    net_copy = copy.deepcopy(net)
    _init_runpp_options(net_copy, algorithm='nr', calculate_voltage_angles=True,
                        init='flat', max_iteration=1, tolerance_mva=1e-8,
                        trafo_model='t', trafo_loading='current',
                        enforce_q_lims=False, check_connectivity=False,
                        voltage_depend_loads=False, consider_line_temperature=False,
                        distributed_slack=False)
    ppc, ppci = _pd2ppc(net_copy)
    Ybus_sparse, _, _ = makeYbus(ppci['baseMVA'], ppci['bus'], ppci['branch'])
    Ybus = Ybus_sparse.toarray()
    bus_lookup = net_copy._pd2ppc_lookups['bus']
    pp_to_ppc = bus_lookup
    n_ppc = Ybus.shape[0]

    ext_set = set(net.ext_grid['bus'].values)
    pv_set = (set(net.gen['bus'].values) if len(net.gen) > 0 else set()) - ext_set
    slack_ppc = [pp_to_ppc[bus_to_idx[b]] for b in ext_set]
    pv_ppc = [pp_to_ppc[bus_to_idx[b]] for b in pv_set]
    pq_ppc = [pp_to_ppc[bus_to_idx[b]] for b in bus_list
              if b not in ext_set and b not in pv_set]
    ptheta_ppc = sorted(set(pv_ppc + pq_ppc))
    pq_ppc = sorted(pq_ppc)

    p_spec = np.zeros(n_ppc)
    for row_idx in net.load.index:
        load = net.load.loc[row_idx]
        p_spec[pp_to_ppc[bus_to_idx[load['bus']]]] -= load['p_mw'] / S_base
    if len(net.gen) > 0:
        for row_idx in net.gen.index:
            gen = net.gen.loc[row_idx]
            p_spec[pp_to_ppc[bus_to_idx[gen['bus']]]] += gen['p_mw'] / S_base
    if len(net.shunt) > 0:
        for row_idx in net.shunt.index:
            shunt = net.shunt.loc[row_idx]
            p_spec[pp_to_ppc[bus_to_idx[shunt['bus']]]] -= shunt['p_mw'] / S_base

    q_spec = np.zeros(n_ppc)
    for row_idx in net.load.index:
        load = net.load.loc[row_idx]
        q_spec[pp_to_ppc[bus_to_idx[load['bus']]]] -= load['q_mvar'] / S_base
    if len(net.shunt) > 0:
        for row_idx in net.shunt.index:
            shunt = net.shunt.loc[row_idx]
            q_spec[pp_to_ppc[bus_to_idx[shunt['bus']]]] -= shunt['q_mvar'] / S_base

    theta = np.zeros(n_ppc)
    V = np.ones(n_ppc)
    for pp_idx, bus in enumerate(bus_list):
        theta[pp_to_ppc[pp_idx]] = np.radians(theta_dc_deg[pp_idx])
    if len(net.gen) > 0:
        for row_idx in net.gen.index:
            gen = net.gen.loc[row_idx]
            V[pp_to_ppc[bus_to_idx[gen['bus']]]] = gen['vm_pu']
    for row_idx in net.ext_grid.index:
        ext = net.ext_grid.loc[row_idx]
        V[pp_to_ppc[bus_to_idx[ext['bus']]]] = ext['vm_pu']

    Bp = -Ybus.imag
    Bp_sub = Bp[np.ix_(ptheta_ppc, ptheta_ppc)]
    Bpp_sub = Bp[np.ix_(pq_ppc, pq_ppc)]

    best_max_mismatch = np.inf
    best_V = V.copy()
    best_theta = theta.copy()
    for iteration in range(n_iter):
        V_complex = V * np.exp(1j * theta)
        S_calc = V_complex * np.conj(Ybus @ V_complex)
        P_calc = S_calc.real
        Q_calc = S_calc.imag

        dP = p_spec[ptheta_ppc] - P_calc[ptheta_ppc]
        dQ = q_spec[pq_ppc] - Q_calc[pq_ppc]

        max_mismatch = max(np.max(np.abs(dP)), np.max(np.abs(dQ)))
        if max_mismatch < best_max_mismatch:
            best_max_mismatch = max_mismatch
            best_V = V.copy()
            best_theta = theta.copy()
        elif max_mismatch > 10 * best_max_mismatch:
            V = best_V; theta = best_theta; break

        if max_mismatch < 1e-6:
            break

        try:
            theta[ptheta_ppc] += np.linalg.solve(Bp_sub, dP / V[ptheta_ppc])
        except np.linalg.LinAlgError:
            pass
        try:
            V[pq_ppc] += np.linalg.solve(Bpp_sub, dQ / V[pq_ppc])
        except np.linalg.LinAlgError:
            pass

    V_complex = V * np.exp(1j * theta)
    S_final = V_complex * np.conj(Ybus @ V_complex)
    Q_final_ppc = S_final.imag

    v_pp = np.ones(n)
    q_pp = np.zeros(n)
    theta_pp = np.zeros(n)
    for pp_idx in range(n):
        ppc_i = pp_to_ppc[pp_idx]
        v_pp[pp_idx] = V[ppc_i]
        q_pp[pp_idx] = -Q_final_ppc[ppc_i] * S_base
        theta_pp[pp_idx] = np.degrees(theta[ppc_i])

    return v_pp, q_pp, theta_pp


def extract_pre_pf_features(net, load_scale, grid_name=''):
    """Extract ONLY pre-PF features."""
    n_bus = len(net.bus)
    bus_info = pd.DataFrame(index=net.bus.index)
    bus_list = net.bus.index.tolist()
    bus_to_idx = {b: i for i, b in enumerate(bus_list)}

    # Structural
    bus_info['vn_kv'] = net.bus['vn_kv'].values
    bus_info['bus_pos_norm'] = np.arange(n_bus) / max(n_bus - 1, 1)
    bus_info['grid_n_bus'] = n_bus

    from_counts = net.line['from_bus'].value_counts()
    to_counts = net.line['to_bus'].value_counts()
    line_degree = from_counts.add(to_counts, fill_value=0)
    if len(net.trafo) > 0:
        hv_counts = net.trafo['hv_bus'].value_counts()
        lv_counts = net.trafo['lv_bus'].value_counts()
        total_degree = line_degree.add(hv_counts.add(lv_counts, fill_value=0), fill_value=0)
    else:
        total_degree = line_degree
    bus_info['n_connections'] = total_degree.reindex(bus_info.index, fill_value=0).astype(int)

    # Load info
    load_per_bus = net.load.groupby('bus').agg(
        nominal_p_mw=('p_mw', 'sum'), nominal_q_mvar=('q_mvar', 'sum'))
    bus_info = bus_info.join(load_per_bus, how='left').fillna(0.0)

    # Generator info
    if len(net.gen) > 0:
        gen_per_bus = net.gen.groupby('bus').agg(
            gen_p_mw=('p_mw', 'sum'), gen_vm_pu=('vm_pu', 'mean'))
        bus_info = bus_info.join(gen_per_bus, how='left').fillna(0.0)
    else:
        bus_info['gen_p_mw'] = 0.0; bus_info['gen_vm_pu'] = 0.0

    # Slack
    ext_grid_buses = set(net.ext_grid['bus'].values)
    bus_info['is_slack'] = bus_info.index.isin(ext_grid_buses).astype(int)
    ext_grid_vm = net.ext_grid.groupby('bus').agg(ext_grid_vm_pu=('vm_pu', 'mean'))
    bus_info = bus_info.join(ext_grid_vm, how='left').fillna(0.0)

    # Shunt
    if len(net.shunt) > 0:
        shunt_per_bus = net.shunt.groupby('bus').agg(
            shunt_p_mw=('p_mw', 'sum'), shunt_q_mvar=('q_mvar', 'sum'))
        bus_info = bus_info.join(shunt_per_bus, how='left').fillna(0.0)
    else:
        bus_info['shunt_p_mw'] = 0.0; bus_info['shunt_q_mvar'] = 0.0
    bus_info['has_shunt'] = ((bus_info['shunt_p_mw'] != 0) | (bus_info['shunt_q_mvar'] != 0)).astype(int)

    # Bus type
    gen_buses = set(net.gen['bus'].values) if len(net.gen) > 0 else set()
    gen_buses |= ext_grid_buses
    bus_info['has_gen'] = bus_info.index.isin(gen_buses).astype(int)
    load_buses = set(net.load['bus'].values)
    bus_info['has_load'] = bus_info.index.isin(load_buses).astype(int)

    pv_buses = set(net.gen['bus'].values) if len(net.gen) > 0 else set()
    bus_type_arr = np.zeros(n_bus, dtype=int)
    bus_type_arr[bus_info.index.isin(pv_buses)] = 1
    bus_type_arr[bus_info.index.isin(ext_grid_buses)] = 2
    bus_info['bus_type'] = bus_type_arr

    # Specified voltage magnitude
    vm_from_ext = bus_info['ext_grid_vm_pu'].values
    vm_from_gen = bus_info['gen_vm_pu'].values
    specified_vm = np.where(vm_from_ext > 0, vm_from_ext,
                   np.where(vm_from_gen > 0, vm_from_gen, 1.0))
    bus_info['specified_vm_pu'] = specified_vm

    # Neighbor features
    adj_v_sum = np.zeros(n_bus)
    adj_count = np.zeros(n_bus)
    for _, line in net.line.iterrows():
        fi, ti = bus_to_idx[line['from_bus']], bus_to_idx[line['to_bus']]
        adj_v_sum[fi] += specified_vm[ti]; adj_count[fi] += 1
        adj_v_sum[ti] += specified_vm[fi]; adj_count[ti] += 1
    if len(net.trafo) > 0:
        for _, trafo in net.trafo.iterrows():
            hi, li = bus_to_idx[trafo['hv_bus']], bus_to_idx[trafo['lv_bus']]
            adj_v_sum[hi] += specified_vm[li]; adj_count[hi] += 1
            adj_v_sum[li] += specified_vm[hi]; adj_count[li] += 1
    bus_info['avg_neighbor_vm'] = np.where(adj_count > 0, adj_v_sum / adj_count, 1.0)

    adj_gen_count = np.zeros(n_bus)
    for _, line in net.line.iterrows():
        fi, ti = bus_to_idx[line['from_bus']], bus_to_idx[line['to_bus']]
        if bus_info.iloc[ti]['has_gen']: adj_gen_count[fi] += 1
        if bus_info.iloc[fi]['has_gen']: adj_gen_count[ti] += 1
    if len(net.trafo) > 0:
        for _, trafo in net.trafo.iterrows():
            hi, li = bus_to_idx[trafo['hv_bus']], bus_to_idx[trafo['lv_bus']]
            if bus_info.iloc[li]['has_gen']: adj_gen_count[hi] += 1
            if bus_info.iloc[hi]['has_gen']: adj_gen_count[li] += 1
    bus_info['frac_gen_neighbors'] = np.where(adj_count > 0, adj_gen_count / adj_count, 0.0)

    # Line impedance features
    if len(net.line) > 0:
        line_x_ohm = net.line['x_ohm_per_km'] * net.line['length_km']
        line_r_ohm = net.line['r_ohm_per_km'] * net.line['length_km']
        line_c_nf = net.line['c_nf_per_km'] * net.line['length_km']
        line_data = pd.DataFrame({
            'from_bus': net.line['from_bus'].values, 'to_bus': net.line['to_bus'].values,
            'x_ohm': line_x_ohm.values, 'r_ohm': line_r_ohm.values, 'c_nf': line_c_nf.values})
        from_agg = line_data.groupby('from_bus').agg(
            sum_x=('x_ohm','sum'), sum_r=('r_ohm','sum'), sum_c=('c_nf','sum'), count=('x_ohm','count'))
        to_agg = line_data.groupby('to_bus').agg(
            sum_x=('x_ohm','sum'), sum_r=('r_ohm','sum'), sum_c=('c_nf','sum'), count=('x_ohm','count'))
        bus_x = from_agg['sum_x'].add(to_agg['sum_x'], fill_value=0).reindex(bus_info.index, fill_value=0)
        bus_r = from_agg['sum_r'].add(to_agg['sum_r'], fill_value=0).reindex(bus_info.index, fill_value=0)
        bus_c = from_agg['sum_c'].add(to_agg['sum_c'], fill_value=0).reindex(bus_info.index, fill_value=0)
        bus_nl = from_agg['count'].add(to_agg['count'], fill_value=0).reindex(bus_info.index, fill_value=0)
        z_base = (bus_info['vn_kv'].values ** 2) / 100.0 + 1e-6
        bus_info['sum_x_pu'] = bus_x.values / z_base
        bus_info['sum_r_pu'] = bus_r.values / z_base
        bus_info['avg_x_pu'] = np.where(bus_nl.values > 0, bus_x.values / bus_nl.values / z_base, 0.0)
        bus_info['sum_b_charging_pu'] = bus_c.values * 1e-9 * 2 * np.pi * 50 / (1.0 / z_base)
    else:
        for c in ['sum_x_pu','sum_r_pu','avg_x_pu','sum_b_charging_pu']:
            bus_info[c] = 0.0

    # Transformer impedance
    if len(net.trafo) > 0:
        trafo_data = pd.DataFrame({
            'hv_bus': net.trafo['hv_bus'].values, 'lv_bus': net.trafo['lv_bus'].values,
            'vk_pct': net.trafo['vk_percent'].values})
        hv_agg = trafo_data.groupby('hv_bus').agg(sum_vk=('vk_pct','sum'))
        lv_agg = trafo_data.groupby('lv_bus').agg(sum_vk=('vk_pct','sum'))
        bus_trafo_vk = hv_agg['sum_vk'].add(lv_agg['sum_vk'], fill_value=0).reindex(bus_info.index, fill_value=0)
        bus_info['trafo_vk_sum'] = bus_trafo_vk.values / 100.0
    else:
        bus_info['trafo_vk_sum'] = 0.0

    # Generator Q limits
    if len(net.gen) > 0:
        gen_q_cols = {}
        if 'min_q_mvar' in net.gen.columns: gen_q_cols['gen_min_q_mvar'] = ('min_q_mvar', 'sum')
        if 'max_q_mvar' in net.gen.columns: gen_q_cols['gen_max_q_mvar'] = ('max_q_mvar', 'sum')
        if gen_q_cols:
            bus_info = bus_info.join(net.gen.groupby('bus').agg(**gen_q_cols), how='left')
    for col in ['gen_min_q_mvar', 'gen_max_q_mvar']:
        if col not in bus_info.columns: bus_info[col] = 0.0
        bus_info[col] = bus_info[col].fillna(0.0)

    # DC Power Flow
    net_dc = copy.deepcopy(net)
    pp.rundcpp(net_dc)
    slack_bus = net.ext_grid['bus'].iloc[0]
    slack_idx = bus_to_idx[slack_bus]
    dc_slack_va = net_dc.res_bus['va_degree'].values[slack_idx]
    bus_info['dc_va_rel'] = net_dc.res_bus['va_degree'].values - dc_slack_va
    dc_va_range = bus_info['dc_va_rel'].max() - bus_info['dc_va_rel'].min()
    bus_info['dc_va_range'] = dc_va_range if dc_va_range > 1e-6 else 1.0

    # FDPF for V, Q, and angles
    theta_dc = net_dc.res_bus['va_degree'].values
    v_approx, q_approx_mvar, fdpf_va_deg = compute_fdpf_vq(net, theta_dc, n_iter=10)
    bus_info['vq_v_approx'] = v_approx
    bus_info['vq_q_approx_mvar'] = q_approx_mvar
    bus_info['q_ac_approx_mvar'] = q_approx_mvar

    # FDPF angles (relative to slack) — much better than DC PF for trafo-heavy grids
    fdpf_slack_va = fdpf_va_deg[slack_idx]
    bus_info['fdpf_va_rel'] = fdpf_va_deg - fdpf_slack_va
    fdpf_va_range = bus_info['fdpf_va_rel'].max() - bus_info['fdpf_va_rel'].min()
    bus_info['fdpf_va_range'] = fdpf_va_range if fdpf_va_range > 1e-6 else 1.0
    bus_info['fdpf_va_norm'] = bus_info['fdpf_va_rel'] / bus_info['fdpf_va_range']

    # Y-bus DC angles (full admittance matrix, includes transformer models)
    ybus_dc_va_deg = compute_ybus_dc_angles(net)
    ybus_dc_slack_va = ybus_dc_va_deg[slack_idx]
    bus_info['ybus_dc_va_rel'] = ybus_dc_va_deg - ybus_dc_slack_va
    ybus_dc_va_range = bus_info['ybus_dc_va_rel'].max() - bus_info['ybus_dc_va_rel'].min()
    bus_info['ybus_dc_va_range'] = ybus_dc_va_range if ybus_dc_va_range > 1e-6 else 1.0
    bus_info['ybus_dc_va_norm'] = bus_info['ybus_dc_va_rel'] / bus_info['ybus_dc_va_range']

    # Transformer-specific features
    bus_info['n_trafos'] = 0
    if len(net.trafo) > 0:
        hv_counts = net.trafo['hv_bus'].value_counts()
        lv_counts = net.trafo['lv_bus'].value_counts()
        trafo_degree = hv_counts.add(lv_counts, fill_value=0)
        bus_info['n_trafos'] = trafo_degree.reindex(bus_info.index, fill_value=0).astype(int)
    bus_info['trafo_frac'] = bus_info['n_trafos'] / (bus_info['n_connections'] + 1e-6)

    # Aggregate stats
    bus_info['total_load_p_mw'] = net.load['p_mw'].sum()
    bus_info['total_gen_p_mw'] = net.gen['p_mw'].sum() if len(net.gen) > 0 else 0.0
    bus_info['load_gen_ratio'] = bus_info['total_load_p_mw'] / max(bus_info['total_gen_p_mw'].iloc[0], 1e-6)
    bus_info['load_scale'] = load_scale
    bus_info['grid_name'] = grid_name

    return bus_info


def solve_grid_at_loading(net_template, load_scale, grid_name=''):
    """Scale loads, extract pre-PF features, solve AC PF for targets."""
    net = copy.deepcopy(net_template)
    net.load['p_mw'] = net_template.load['p_mw'] * load_scale
    net.load['q_mvar'] = net_template.load['q_mvar'] * load_scale

    features = extract_pre_pf_features(net, load_scale, grid_name)

    pp.runpp(net)
    slack_bus = net.ext_grid['bus'].iloc[0]
    slack_idx = list(net.bus.index).index(slack_bus)
    slack_va = net.res_bus['va_degree'].values[slack_idx]

    features['vm_pu'] = net.res_bus['vm_pu'].values
    features['va_degree'] = net.res_bus['va_degree'].values
    features['va_rel'] = net.res_bus['va_degree'].values - slack_va
    features['p_mw'] = net.res_bus['p_mw'].values
    features['q_mvar'] = net.res_bus['q_mvar'].values

    return features

## 2. Normalization & Target Encoding

In [ ]:
def add_normalized_features(df):
    """Add per-grid normalized features and targets."""
    out = df.copy()
    load_total = df['total_load_p_mw'].abs() + 1e-6
    gen_total  = df['total_gen_p_mw'].abs() + 1e-6

    out['nominal_p_pu']    = df['nominal_p_mw']    / load_total
    out['nominal_q_pu']    = df['nominal_q_mvar']  / load_total
    out['gen_p_pu']        = df['gen_p_mw']        / gen_total
    out['shunt_p_pu']      = df['shunt_p_mw']      / load_total
    out['shunt_q_pu']      = df['shunt_q_mvar']    / load_total
    out['gen_min_q_pu']    = df['gen_min_q_mvar']  / load_total
    out['gen_max_q_pu']    = df['gen_max_q_mvar']  / load_total
    out['net_p_pu']        = (df['gen_p_mw'] - df['nominal_p_mw']) / load_total
    out['net_q_specified_pu'] = (-df['nominal_q_mvar'] + df['shunt_q_mvar']) / load_total
    out['log_n_bus']       = np.log1p(df['grid_n_bus'])

    dc_va_range = df['dc_va_range'].abs() + 1e-6
    out['dc_va_norm']      = df['dc_va_rel'] / dc_va_range

    out['p_x_product']     = out['net_p_pu'] * df['avg_x_pu']
    out['v_drop_proxy']    = out['nominal_p_pu'] * df['sum_r_pu'] + out['nominal_q_pu'] * df['sum_x_pu']

    out['vq_q_approx_pu'] = df['vq_q_approx_mvar'] / load_total
    out['q_ac_approx_pu'] = df['q_ac_approx_mvar'] / load_total

    out['p_specified_mw']   = df['nominal_p_mw'] - df['gen_p_mw'] + df['shunt_p_mw']
    out['q_specified_mvar'] = df['nominal_q_mvar'] + df['shunt_q_mvar']

    # TARGETS
    out['target_vm']           = df['vm_pu'] - df['vq_v_approx']
    dc_va_range = df['dc_va_range'].abs() + 1e-6
    out['target_va_norm']      = (df['va_rel'] - df['dc_va_rel']) / dc_va_range
    out['target_p_pu']         = (df['p_mw'] - out['p_specified_mw']) / load_total
    out['target_q_pu']         = (df['q_mvar'] - df['q_ac_approx_mvar']) / load_total

    return out


def reconstruct_predictions(preds, test_norm_df, target_name):
    """Reconstruct absolute values from residual predictions + physics baselines."""
    load_total = test_norm_df['total_load_p_mw'].abs().values + 1e-6

    if target_name == 'target_vm':
        recon = test_norm_df['vq_v_approx'].values + preds
        gen_mask = test_norm_df['has_gen'].values.astype(bool)
        recon[gen_mask] = test_norm_df['specified_vm_pu'].values[gen_mask]
        return recon

    elif target_name == 'target_va_norm':
        dc_va_range = test_norm_df['dc_va_range'].abs().values + 1e-6
        recon = test_norm_df['dc_va_rel'].values + preds * dc_va_range
        slack_mask = test_norm_df['is_slack'].values.astype(bool)
        recon[slack_mask] = 0.0
        return recon

    elif target_name == 'target_p_pu':
        return test_norm_df['p_specified_mw'].values + preds * load_total

    elif target_name == 'target_q_pu':
        return test_norm_df['q_ac_approx_mvar'].values + preds * load_total

    else:
        return preds

## 3. Grid Configuration

Training: case9, case14, case30, case39, case57, case118 (15 scales: 0.50–1.30)
Test: case300 at scale=1.0 only

In [4]:
train_grid_loaders = {
    'case9':  pn.case9,
    'case14': pn.case14,
    'case30': pn.case30,
    'case39': pn.case39,
    'case57': pn.case57,
    'case118': pn.case118,
}

test_grid_loaders = {
    'case300': pn.case300,
}

n_load_conditions = 15
train_scales = np.linspace(0.50, 1.30, n_load_conditions).tolist()
test_scales = [1.0]

print(f"Training grids: {list(train_grid_loaders.keys())}")
print(f"Test grid (UNSEEN): {list(test_grid_loaders.keys())}")
print(f"Training scales ({n_load_conditions}): {[f'{s:.2f}' for s in train_scales]}")
print(f"Test scales: {test_scales}")

Training grids: ['case9', 'case14', 'case30', 'case39', 'case57', 'case118']
Test grid (UNSEEN): ['case300']
Training scales (15): ['0.50', '0.56', '0.61', '0.67', '0.73', '0.79', '0.84', '0.90', '0.96', '1.01', '1.07', '1.13', '1.19', '1.24', '1.30']
Test scales: [1.0]


## 4. Generate Training Data

In [5]:
train_nets = {}
for name, loader in train_grid_loaders.items():
    net = loader()
    train_nets[name] = net
    print(f"  {name}: {len(net.bus)} buses, {len(net.line)} lines, "
          f"{len(net.gen)} gens, {len(net.load)} loads, {len(net.trafo)} trafos, {len(net.shunt)} shunts")

train_frames = []
for grid_name, net_template in train_nets.items():
    n_ok = 0
    for s in train_scales:
        try:
            df = solve_grid_at_loading(net_template, s, grid_name)
            train_frames.append(df)
            n_ok += 1
        except Exception as e:
            print(f"  WARN: {grid_name} scale={s:.2f} failed: {e}")
    print(f"  {grid_name}: {n_ok}/{len(train_scales)} scales converged")

train_df = pd.concat(train_frames, ignore_index=True)
print(f"\nTotal training rows: {len(train_df)}")
print(train_df['grid_name'].value_counts().to_string())

  case9: 9 buses, 9 lines, 2 gens, 3 loads, 0 trafos, 0 shunts
  case14: 14 buses, 15 lines, 4 gens, 11 loads, 5 trafos, 1 shunts
  case30: 30 buses, 41 lines, 5 gens, 20 loads, 0 trafos, 2 shunts
  case39: 39 buses, 35 lines, 9 gens, 21 loads, 11 trafos, 0 shunts
  case57: 57 buses, 63 lines, 6 gens, 42 loads, 17 trafos, 3 shunts
  case118: 118 buses, 173 lines, 53 gens, 99 loads, 13 trafos, 14 shunts
  case9: 15/15 scales converged
  case14: 15/15 scales converged
  case30: 15/15 scales converged
  WARN: case39 scale=1.30 failed: Power Flow nr did not converge after 10 iterations!
  case39: 14/15 scales converged
  case57: 15/15 scales converged
  case118: 15/15 scales converged

Total training rows: 3966
grid_name
case118    1770
case57      855
case39      546
case30      450
case14      210
case9       135


## 5. Generate Test Data (IEEE 300, scale=1.0)

In [6]:
test_nets = {}
test_frames = []

for name, loader in test_grid_loaders.items():
    net = loader()
    test_nets[name] = net
    print(f"  {name}: {len(net.bus)} buses, {len(net.line)} lines, "
          f"{len(net.trafo)} trafos, {len(net.gen)} gens, {len(net.load)} loads, {len(net.shunt)} shunts")

    for s in test_scales:
        try:
            df = solve_grid_at_loading(net, s, name)
            test_frames.append(df)
        except Exception as e:
            print(f"  WARN: {name} scale={s} failed: {e}")
    print(f"  Solved at {len(test_scales)} scale(s)")

test_df = pd.concat(test_frames, ignore_index=True)
print(f"\nTotal test rows: {len(test_df)}")

  case300: 300 buses, 283 lines, 128 trafos, 68 gens, 193 loads, 29 shunts
  Solved at 1 scale(s)

Total test rows: 300


## 6. Feature Selection & Baseline Diagnostics

In [7]:
train_norm = add_normalized_features(train_df)
test_norm  = add_normalized_features(test_df)

feature_cols = [
    # Structural
    'bus_pos_norm', 'log_n_bus', 'n_connections',
    # Specified power injections (normalized)
    'nominal_p_pu', 'nominal_q_pu', 'gen_p_pu',
    'net_p_pu', 'net_q_specified_pu',
    'shunt_p_pu', 'shunt_q_pu',
    # Bus type & voltage setpoints
    'gen_vm_pu', 'is_slack', 'ext_grid_vm_pu',
    'has_gen', 'has_load', 'has_shunt', 'bus_type',
    'specified_vm_pu',
    # Neighbor voltage support
    'avg_neighbor_vm', 'frac_gen_neighbors',
    # Network impedance
    'sum_x_pu', 'sum_r_pu', 'avg_x_pu',
    'sum_b_charging_pu', 'trafo_vk_sum',
    # Generator Q limits
    'gen_min_q_pu', 'gen_max_q_pu',
    # DC PF angles
    'dc_va_rel', 'dc_va_norm', 'dc_va_range',
    # FDPF approximation (angles, V, Q)
    'fdpf_va_rel', 'fdpf_va_norm', 'fdpf_va_range',
    # Y-bus DC angles (full admittance matrix)
    'ybus_dc_va_rel', 'ybus_dc_va_norm', 'ybus_dc_va_range',
    'vq_v_approx', 'vq_q_approx_pu',
    'q_ac_approx_pu',
    # Transformer topology
    'n_trafos', 'trafo_frac',
    # System context & physics interactions
    'load_gen_ratio', 'load_scale',
    'p_x_product', 'v_drop_proxy',
]

target_cols = ['target_vm', 'target_va_norm', 'target_p_pu', 'target_q_pu']
original_target_cols  = ['vm_pu', 'va_rel', 'p_mw', 'q_mvar']
original_target_labels = ['vm_pu', 'va_degree (rel)', 'p_mw', 'q_mvar']

X_train = train_norm[feature_cols]
Y_train = train_norm[target_cols]
X_test  = test_norm[feature_cols]

# Baselines
gen_mask = test_norm['has_gen'].values.astype(bool)
baseline_vm = test_norm['specified_vm_pu'].values.copy()
baseline_vm[~gen_mask] = test_norm.loc[~gen_mask, 'vq_v_approx'].values
baseline_va_dc = test_norm['dc_va_rel'].values.copy()
baseline_va_dc[test_norm['is_slack'].values.astype(bool)] = 0.0
baseline_va_fdpf = test_norm['fdpf_va_rel'].values.copy()
baseline_va_fdpf[test_norm['is_slack'].values.astype(bool)] = 0.0
baseline_p = test_norm['p_specified_mw'].values
baseline_q = test_norm['q_specified_mvar'].values

print(f"X_train: {X_train.shape}  ({len(feature_cols)} features)")
print(f"Y_train: {Y_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\n--- Physics Baseline R² (no model) ---")
print(f"  vm_pu (FDPF V + PV override):  {r2_score(test_df['vm_pu'], baseline_vm):.4f}")
print(f"  va_degree (DC PF):             {r2_score(test_df['va_rel'], baseline_va_dc):.4f}")
print(f"  va_degree (FDPF):              {r2_score(test_df['va_rel'], baseline_va_fdpf):.4f}")
print(f"  p_mw (specification):          {r2_score(test_df['p_mw'], baseline_p):.4f}")
print(f"  q_mvar (specification):        {r2_score(test_df['q_mvar'], baseline_q):.4f}")
print(f"  q_mvar (FDPF):                 {r2_score(test_df['q_mvar'], test_df['q_ac_approx_mvar']):.4f}")

# Y-bus DC angles
ybus_dc_va_baseline = test_norm['ybus_dc_va_rel'].values.copy()
ybus_dc_va_baseline[test_norm['is_slack'].values.astype(bool)] = 0.0
print(f"  va_degree (Y-bus DC):          {r2_score(test_df['va_rel'], ybus_dc_va_baseline):.4f}")

X_train: (3966, 45)  (45 features)
Y_train: (3966, 4)
X_test:  (300, 45)

--- Physics Baseline R² (no model) ---
  vm_pu (FDPF V + PV override):  0.8031
  va_degree (DC PF):             -0.8347
  va_degree (FDPF):              -4.7934
  p_mw (specification):          0.9899
  q_mvar (specification):        0.1869
  q_mvar (FDPF):                 0.7983
  va_degree (Y-bus DC):          -32094.6663


In [10]:
actual_va = test_df['va_degree'].values
dc_va = test_df['dc_va_rel'].values
fdpf_va = test_df['fdpf_va_rel'].values

print("--- Angle Analysis ---")
print(f"  Actual va range: [{actual_va.min():.1f}, {actual_va.max():.1f}], span={actual_va.max()-actual_va.min():.1f}")
print(f"  DC PF va range:  [{dc_va.min():.1f}, {dc_va.max():.1f}], span={dc_va.max()-dc_va.min():.1f}")
print(f"  FDPF va range:   [{fdpf_va.min():.1f}, {fdpf_va.max():.1f}], span={fdpf_va.max()-fdpf_va.min():.1f}")

from scipy.stats import pearsonr, linregress
print(f"\n  DC PF corr:   {pearsonr(actual_va, dc_va)[0]:.4f}")
print(f"  FDPF corr:    {pearsonr(actual_va, fdpf_va)[0]:.4f}")

# Linear fit: actual = slope * dc + intercept
slope, intercept, r, p, se = linregress(dc_va, actual_va)
print(f"\n  Linear fit actual = {slope:.4f} * dc_pf + {intercept:.4f}")
print(f"  R² of linear fit: {r**2:.4f}")
dc_scaled = slope * dc_va + intercept
from sklearn.metrics import r2_score
print(f"  R² after scaling: {r2_score(actual_va, dc_scaled):.4f}")
print(f"  MAE after scaling: {np.mean(np.abs(actual_va - dc_scaled)):.2f}")

# Check per-grid scaling in training data
print("\n--- Per-Grid DC Angle Scaling (training data) ---")
for gname in train_df['grid_name'].unique():
    mask = train_df['grid_name'] == gname
    g_actual = train_df.loc[mask, 'va_degree'].values
    g_dc = train_df.loc[mask, 'dc_va_rel'].values
    s, i, r2, _, _ = linregress(g_dc, g_actual)
    corr = pearsonr(g_actual, g_dc)[0]
    print(f"  {gname:>8s}: slope={s:.4f}, intercept={i:.4f}, R²={r2**2:.4f}, corr={corr:.4f}")

print(f"\n  case300:   slope={slope:.4f}, intercept={intercept:.4f}, R²={r**2:.4f}")

# What if we predict the RATIO actual/dc for non-zero dc?
mask_nonzero = np.abs(dc_va) > 0.1
ratios = actual_va[mask_nonzero] / dc_va[mask_nonzero]
print(f"\n--- Angle Ratio Analysis (|dc|>0.1) ---")
print(f"  Ratio mean: {ratios.mean():.4f}, std: {ratios.std():.4f}")
print(f"  Ratio range: [{ratios.min():.4f}, {ratios.max():.4f}]")

# Check if the slope varies across bus types
print("\n--- DC Angle Fit by Bus Type (case300) ---")
for bt in sorted(test_df['bus_type'].unique()):
    mask = test_df['bus_type'] == bt
    g_actual = actual_va[mask]
    g_dc = dc_va[mask]
    s, i, r2, _, _ = linregress(g_dc, g_actual)
    print(f"  type={bt}: n={mask.sum()}, slope={s:.4f}, intercept={i:.4f}, R²={r2**2:.4f}")

print("\n--- FDPF V Analysis ---")
fdpf_v = test_df['vq_v_approx'].values
actual_v = test_df['vm_pu'].values
print(f"  Overall V MAE: {np.mean(np.abs(actual_v - fdpf_v)):.4f}")
mask_pq = test_df['bus_type'] == 0
print(f"  PQ bus V MAE:  {np.mean(np.abs(actual_v[mask_pq] - fdpf_v[mask_pq])):.4f}")
print(f"  PQ bus V R²:   {r2_score(actual_v[mask_pq], fdpf_v[mask_pq]):.4f}")

--- Angle Analysis ---
  Actual va range: [-40.0, 35.6], span=75.6
  DC PF va range:  [-19.5, 56.6], span=76.1
  FDPF va range:   [-21.6, 91.1], span=112.7

  DC PF corr:   0.9780
  FDPF corr:    0.9187

  Linear fit actual = 0.9278 * dc_pf + -17.1310
  R² of linear fit: 0.9564
  R² after scaling: 0.9564
  MAE after scaling: 2.17

--- Per-Grid DC Angle Scaling (training data) ---
     case9: slope=1.0093, intercept=-0.2874, R²=0.9995, corr=0.9997
    case14: slope=0.9655, intercept=0.1661, R²=0.9974, corr=0.9987
    case30: slope=1.0587, intercept=0.1069, R²=0.9976, corr=0.9988
    case39: slope=1.1711, intercept=-2.8526, R²=0.9834, corr=0.9917
    case57: slope=1.1339, intercept=0.4106, R²=0.9937, corr=0.9968
   case118: slope=1.0629, intercept=27.3681, R²=0.9965, corr=0.9983

  case300:   slope=0.9278, intercept=-17.1310, R²=0.9564

--- Angle Ratio Analysis (|dc|>0.1) ---
  Ratio mean: 2.2949, std: 15.5145
  Ratio range: [-99.7916, 145.4507]

--- DC Angle Fit by Bus Type (case300) --

## 7. Train TabPFN & Evaluate

In [8]:
from tabpfn import TabPFNRegressor

BATCH_SIZE = 500

def batched_predict(reg, X, batch_size=BATCH_SIZE):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(reg.predict(X.iloc[i:i+batch_size]))
    return np.concatenate(preds)

results = {}

gen_mask_train = train_norm['has_gen'].values.astype(bool)
gen_mask_test  = test_norm['has_gen'].values.astype(bool)
pq_mask_test   = ~gen_mask_test
load_total_train = train_norm['total_load_p_mw'].abs().values + 1e-6
load_total_test  = test_norm['total_load_p_mw'].abs().values + 1e-6

# StandardScaler for Ridge models
scaler_all = StandardScaler()
X_train_s = scaler_all.fit_transform(X_train)
X_test_s  = scaler_all.transform(X_test)

# ====================================================================
# VM: Compare TabPFN residual vs Ridge residual vs physics baseline
# ====================================================================
print("="*60)
print("VM approaches:")
actual_vm = test_df['vm_pu'].values

# Physics baseline (FDPF V + PV override)
vm_physics = test_norm['vq_v_approx'].values.copy()
vm_physics[gen_mask_test] = test_norm.loc[gen_mask_test, 'specified_vm_pu'].values
r2_vm_physics = r2_score(actual_vm, vm_physics)

# TabPFN residual
reg_vm = TabPFNRegressor()
reg_vm.fit(X_train, Y_train['target_vm'])
vm_tabpfn_resid = batched_predict(reg_vm, X_test)
vm_tabpfn = test_norm['vq_v_approx'].values + vm_tabpfn_resid
vm_tabpfn[gen_mask_test] = test_norm.loc[gen_mask_test, 'specified_vm_pu'].values
r2_vm_tabpfn = r2_score(actual_vm, vm_tabpfn)

# Ridge residual
ridge_vm = RidgeCV(alphas=np.logspace(-2, 6, 50))
ridge_vm.fit(X_train_s, Y_train['target_vm'])
vm_ridge_resid = ridge_vm.predict(X_test_s)
vm_ridge = test_norm['vq_v_approx'].values + vm_ridge_resid
vm_ridge[gen_mask_test] = test_norm.loc[gen_mask_test, 'specified_vm_pu'].values
r2_vm_ridge = r2_score(actual_vm, vm_ridge)

# Ridge on absolute vm_pu
ridge_vm_abs = RidgeCV(alphas=np.logspace(-2, 6, 50))
ridge_vm_abs.fit(X_train_s, train_df['vm_pu'].values)
vm_ridge_abs = ridge_vm_abs.predict(X_test_s)
vm_ridge_abs[gen_mask_test] = test_norm.loc[gen_mask_test, 'specified_vm_pu'].values
r2_vm_ridge_abs = r2_score(actual_vm, vm_ridge_abs)

vm_approaches = {
    'physics': (r2_vm_physics, vm_physics),
    'tabpfn': (r2_vm_tabpfn, vm_tabpfn),
    'ridge_resid': (r2_vm_ridge, vm_ridge),
    'ridge_abs': (r2_vm_ridge_abs, vm_ridge_abs),
}
for name, (r2, _) in vm_approaches.items():
    print(f"  {name:16s} R²={r2:.4f}")
best_vm_name = max(vm_approaches, key=lambda k: vm_approaches[k][0])
best_vm_r2, best_vm_pred = vm_approaches[best_vm_name]
print(f"  → Best: {best_vm_name} (R²={best_vm_r2:.4f})")

# ====================================================================
# VA: Compare approaches — this is the hardest target
# ====================================================================
print(f"\n{'='*60}")
print("VA approaches:")
actual_va = test_df['va_rel'].values
ybus_dc_va_range_test = test_norm['ybus_dc_va_range'].abs().values + 1e-6

# DC PF baseline (pandapower)
va_dc = test_norm['dc_va_rel'].values.copy()
va_dc[test_norm['is_slack'].values.astype(bool)] = 0.0
r2_va_dc = r2_score(actual_va, va_dc)

# Y-bus DC baseline
va_ybus_dc = test_norm['ybus_dc_va_rel'].values.copy()
va_ybus_dc[test_norm['is_slack'].values.astype(bool)] = 0.0
r2_va_ybus_dc = r2_score(actual_va, va_ybus_dc)

# TabPFN on Y-bus DC residual (normalized)
reg_va = TabPFNRegressor()
reg_va.fit(X_train, Y_train['target_va_norm'])
va_tabpfn_resid = batched_predict(reg_va, X_test)
va_tabpfn = test_norm['ybus_dc_va_rel'].values + va_tabpfn_resid * ybus_dc_va_range_test
va_tabpfn[test_norm['is_slack'].values.astype(bool)] = 0.0
r2_va_tabpfn = r2_score(actual_va, va_tabpfn)

# Ridge on Y-bus DC residual (normalized)
ridge_va = RidgeCV(alphas=np.logspace(-2, 6, 50))
ridge_va.fit(X_train_s, Y_train['target_va_norm'])
va_ridge_resid = ridge_va.predict(X_test_s)
va_ridge = test_norm['ybus_dc_va_rel'].values + va_ridge_resid * ybus_dc_va_range_test
va_ridge[test_norm['is_slack'].values.astype(bool)] = 0.0
r2_va_ridge = r2_score(actual_va, va_ridge)

# Ridge on absolute va_rel
ridge_va_abs = RidgeCV(alphas=np.logspace(-2, 6, 50))
ridge_va_abs.fit(X_train_s, train_df['va_rel'].values)
va_ridge_abs = ridge_va_abs.predict(X_test_s)
va_ridge_abs[test_norm['is_slack'].values.astype(bool)] = 0.0
r2_va_ridge_abs = r2_score(actual_va, va_ridge_abs)

# TabPFN on absolute va_rel (normalized by Y-bus DC range)
reg_va_abs = TabPFNRegressor()
va_rel_train_norm = train_df['va_rel'].values / (train_norm['ybus_dc_va_range'].abs().values + 1e-6)
reg_va_abs.fit(X_train, va_rel_train_norm)
va_tabpfn_abs_norm = batched_predict(reg_va_abs, X_test)
va_tabpfn_abs = va_tabpfn_abs_norm * ybus_dc_va_range_test
va_tabpfn_abs[test_norm['is_slack'].values.astype(bool)] = 0.0
r2_va_tabpfn_abs = r2_score(actual_va, va_tabpfn_abs)

va_approaches = {
    'dc_pf': (r2_va_dc, va_dc),
    'ybus_dc': (r2_va_ybus_dc, va_ybus_dc),
    'tabpfn_resid': (r2_va_tabpfn, va_tabpfn),
    'ridge_resid': (r2_va_ridge, va_ridge),
    'ridge_abs': (r2_va_ridge_abs, va_ridge_abs),
    'tabpfn_abs': (r2_va_tabpfn_abs, va_tabpfn_abs),
}
for name, (r2, _) in va_approaches.items():
    print(f"  {name:16s} R²={r2:.4f}")
best_va_name = max(va_approaches, key=lambda k: va_approaches[k][0])
best_va_r2, best_va_pred = va_approaches[best_va_name]
print(f"  → Best: {best_va_name} (R²={best_va_r2:.4f})")

# ====================================================================
# P: TabPFN residual vs specification baseline
# ====================================================================
print(f"\n{'='*60}")
print("P approaches:")
actual_p = test_df['p_mw'].values

# Specification baseline
p_spec = test_norm['p_specified_mw'].values
r2_p_spec = r2_score(actual_p, p_spec)

# TabPFN residual
reg_p = TabPFNRegressor()
reg_p.fit(X_train, Y_train['target_p_pu'])
p_tabpfn_resid = batched_predict(reg_p, X_test)
p_tabpfn = p_spec + p_tabpfn_resid * load_total_test
r2_p_tabpfn = r2_score(actual_p, p_tabpfn)

# Ridge residual
ridge_p = RidgeCV(alphas=np.logspace(-2, 6, 50))
ridge_p.fit(X_train_s, Y_train['target_p_pu'])
p_ridge_resid = ridge_p.predict(X_test_s)
p_ridge = p_spec + p_ridge_resid * load_total_test
r2_p_ridge = r2_score(actual_p, p_ridge)

p_approaches = {
    'spec': (r2_p_spec, p_spec),
    'tabpfn': (r2_p_tabpfn, p_tabpfn),
    'ridge': (r2_p_ridge, p_ridge),
}
for name, (r2, _) in p_approaches.items():
    print(f"  {name:16s} R²={r2:.4f}")
best_p_name = max(p_approaches, key=lambda k: p_approaches[k][0])
best_p_r2, best_p_pred = p_approaches[best_p_name]
print(f"  → Best: {best_p_name} (R²={best_p_r2:.4f})")

# ====================================================================
# Q: Split strategy (PQ = spec, PV/slack = best approach)
# ====================================================================
print(f"\n{'='*60}")
print(f"Q PV/slack approaches (gen buses only, {gen_mask_test.sum()} test):")
actual_q_gen_test = test_df.loc[gen_mask_test, 'q_mvar'].values
q_vq_gen_test = test_df.loc[gen_mask_test, 'vq_q_approx_mvar'].values
r2_vq_only = r2_score(actual_q_gen_test, q_vq_gen_test)

X_train_gen = X_train[gen_mask_train].copy()
X_test_gen  = X_test[gen_mask_test].copy()
scaler_gen = StandardScaler()
X_train_gen_s = scaler_gen.fit_transform(X_train_gen)
X_test_gen_s  = scaler_gen.transform(X_test_gen)

# Ridge on absolute Q
y_q_gen_train = train_df.loc[gen_mask_train, 'q_mvar'].values
ridge_q = RidgeCV(alphas=np.logspace(-2, 6, 50))
ridge_q.fit(X_train_gen_s, y_q_gen_train)
q_ridge_gen = ridge_q.predict(X_test_gen_s)
r2_q_ridge = r2_score(actual_q_gen_test, q_ridge_gen)

# Ridge on FDPF residual
y_q_resid = y_q_gen_train - train_df.loc[gen_mask_train, 'vq_q_approx_mvar'].values
ridge_q_resid = RidgeCV(alphas=np.logspace(-2, 6, 50))
ridge_q_resid.fit(X_train_gen_s, y_q_resid)
q_ridge_resid = q_vq_gen_test + ridge_q_resid.predict(X_test_gen_s)
r2_q_ridge_resid = r2_score(actual_q_gen_test, q_ridge_resid)

# TabPFN on absolute Q (pu)
reg_q_gen = TabPFNRegressor()
reg_q_gen.fit(X_train_gen, y_q_gen_train / load_total_train[gen_mask_train])
q_tabpfn_gen = batched_predict(reg_q_gen, X_test_gen) * load_total_test[gen_mask_test]
r2_q_tabpfn = r2_score(actual_q_gen_test, q_tabpfn_gen)

# TabPFN on FDPF residual (pu)
reg_q_resid = TabPFNRegressor()
reg_q_resid.fit(X_train_gen, y_q_resid / load_total_train[gen_mask_train])
q_tabpfn_resid = q_vq_gen_test + batched_predict(reg_q_resid, X_test_gen) * load_total_test[gen_mask_test]
r2_q_tabpfn_resid = r2_score(actual_q_gen_test, q_tabpfn_resid)

q_approaches = {
    'vq_only': (r2_vq_only, q_vq_gen_test),
    'ridge_abs': (r2_q_ridge, q_ridge_gen),
    'ridge_resid': (r2_q_ridge_resid, q_ridge_resid),
    'tabpfn_abs': (r2_q_tabpfn, q_tabpfn_gen),
    'tabpfn_resid': (r2_q_tabpfn_resid, q_tabpfn_resid),
}
for name, (r2, _) in q_approaches.items():
    print(f"  {name:16s} R²={r2:.4f}")
best_q_name = max(q_approaches, key=lambda k: q_approaches[k][0])
best_q_r2, best_q_pred_gen = q_approaches[best_q_name]
print(f"  → Best: {best_q_name} (R²={best_q_r2:.4f})")

# Assemble Q: PQ from spec, PV/slack from best
q_recon = np.zeros(len(test_df))
q_recon[pq_mask_test] = test_norm.loc[pq_mask_test, 'q_specified_mvar'].values
q_recon[gen_mask_test] = best_q_pred_gen
r2_q_full = r2_score(test_df['q_mvar'].values, q_recon)
print(f"  Full Q R² (PQ spec + PV {best_q_name}): {r2_q_full:.4f}")

# ====================================================================
# COLLECT RESULTS
# ====================================================================
for label, ot, recon_data in [
    ('vm_pu', 'vm_pu', best_vm_pred),
    ('va_degree (rel)', 'va_rel', best_va_pred),
    ('p_mw', 'p_mw', best_p_pred),
    ('q_mvar', 'q_mvar', q_recon),
]:
    actual = test_df[ot].values
    r2  = r2_score(actual, recon_data)
    mae = mean_absolute_error(actual, recon_data)
    mse = mean_squared_error(actual, recon_data)
    results[label] = {'r2': r2, 'mae': mae, 'mse': mse, 'preds': recon_data, 'actual': actual, 'ot': ot}

print(f"\n\n{'='*60}")
print(f"SUMMARY: Cross-grid prediction on case300 (unseen, scale=1.0)")
print(f"{'='*60}")
all_pass = True
for label in original_target_labels:
    r = results[label]
    ok = r['r2'] > 0.85
    if not ok: all_pass = False
    method = {'vm_pu': best_vm_name, 'va_degree (rel)': best_va_name,
              'p_mw': best_p_name, 'q_mvar': best_q_name}[label]
    print(f"  {'✓' if ok else '✗'} {label:16s}  R²={r['r2']:.4f}  MAE={r['mae']:.6f}  ({method})")
print(f"\n  All R² > 0.85: {'YES ✓' if all_pass else 'NO ✗'}")

VM approaches:
  physics          R²=0.8031
  tabpfn           R²=0.8309
  ridge_resid      R²=0.8308
  ridge_abs        R²=0.8307
  → Best: tabpfn (R²=0.8309)

VA approaches:
  dc_pf            R²=-0.8347
  tabpfn_resid     R²=-0.9403
  ridge_resid      R²=-0.8206
  ridge_abs        R²=-1.7989
  tabpfn_abs       R²=-0.5252
  → Best: tabpfn_abs (R²=-0.5252)

P approaches:
  spec             R²=0.9899
  tabpfn           R²=0.8986
  ridge            R²=-6.4756
  → Best: spec (R²=0.9899)

Q PV/slack approaches (gen buses only, 69 test):
  vq_only          R²=0.6858
  ridge_abs        R²=-1.1470
  ridge_resid      R²=0.6789
  tabpfn_abs       R²=-1.5911
  tabpfn_resid     R²=-0.8710
  → Best: vq_only (R²=0.6858)
  Full Q R² (PQ spec + PV vq_only): 0.8243


SUMMARY: Cross-grid prediction on case300 (unseen, scale=1.0)
  ✗ vm_pu             R²=0.8309  MAE=0.009057  (tabpfn)
  ✗ va_degree (rel)   R²=-0.5252  MAE=15.068131  (tabpfn_abs)
  ✓ p_mw              R²=0.9899  MAE=2.648077  (spec)
  ✗

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for i, label in enumerate(original_target_labels):
    res = results[label]
    actual = res['actual']
    preds = res['preds']

    ax = axes[0, i]
    ax.scatter(actual, preds, alpha=0.4, s=10, c='steelblue')
    lims = [min(actual.min(), preds.min()), max(actual.max(), preds.max())]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel(f'Actual {label}')
    ax.set_ylabel(f'Predicted {label}')
    ax.set_title(f'{label}  (R² = {res["r2"]:.4f})')

    ax = axes[1, i]
    residuals = actual - preds
    ax.hist(residuals, bins=40, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Residual')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} residuals (MAE = {res["mae"]:.4f})')

plt.suptitle('Cross-Grid: Train [case9..case118] → Test case300 (unseen, scale=1.0)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Error Analysis by Bus Role

In [ ]:
def get_bus_role(row):
    roles = []
    if row['is_slack']:   roles.append('Slack')
    elif row['has_gen']:  roles.append('Generator')
    if row['has_load']:   roles.append('Load')
    if row['has_shunt']:  roles.append('Shunt')
    if not roles:         roles.append('Transit')
    return '+'.join(roles)

mask_1 = test_df['load_scale'] == 1.0
err_df = pd.DataFrame()
err_df['is_slack'] = test_df.loc[mask_1, 'is_slack'].values
err_df['has_gen']  = test_df.loc[mask_1, 'has_gen'].values
err_df['has_load'] = test_df.loc[mask_1, 'has_load'].values
err_df['has_shunt'] = test_df.loc[mask_1, 'has_shunt'].values
err_df['n_connections'] = test_df.loc[mask_1, 'n_connections'].values
err_df['role'] = err_df.apply(get_bus_role, axis=1)

for label in original_target_labels:
    ot = results[label]['ot']
    actual_1 = test_df.loc[mask_1, ot].values
    preds_1  = results[label]['preds'][mask_1.values]
    err_df[f'{label}_err'] = np.abs(actual_1 - preds_1)
err_df['total_err'] = sum(err_df[f'{label}_err'] for label in original_target_labels)

role_stats = err_df.groupby('role').agg(
    count=('total_err', 'count'),
    mean_total=('total_err', 'mean'),
    **{f'mean_{label}': (f'{label}_err', 'mean') for label in original_target_labels}
).sort_values('mean_total', ascending=False)
print("Mean Absolute Error by Bus Role (scale=1.0):")
print(role_stats.to_string())

roles_sorted = role_stats.index.tolist()
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, label in enumerate(original_target_labels):
    ax = axes[i]
    data = [err_df[err_df['role'] == r][f'{label}_err'].values for r in roles_sorted]
    bp = ax.boxplot(data, tick_labels=roles_sorted, vert=True, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
    ax.set_title(f'{label} |error|')
    ax.set_ylabel('Absolute Error')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Error by Bus Role — case300 (unseen, scale=1.0)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()